In [ ]:
import langdetect
import numpy as np
import pandas as pd

import html
import re

Preprocess smartvote.

In [2]:
smartvote_de = pd.read_csv("smartvote_data/23_ch_nr-questions.de.csv")
smartvote_fr = pd.read_csv("smartvote_data/23_ch_nr-questions.fr.csv")
smartvote_it = pd.read_csv("smartvote_data/23_ch_nr-questions.it.csv")

In [3]:
smartvote = pd.concat([smartvote_de, smartvote_fr, smartvote_it])

Drop unneeded columns.

In [4]:
cols_to_drop = ['ID_election', 'category', 'tag_1', 'tag_2', 'tag_3',
                'tag_4', 'tag_5',
                'type', 'rapide', 'cleavage_1', 'cleavage_2',
                'cleavage_3', 'cleavage_4', 'cleavage_5',
                'cleavage_6', 'cleavage_7', 'cleavage_8']
smartvote = smartvote.drop(cols_to_drop, axis=1)

Transform language codes for later merging.

In [6]:
language_mapping = {0: "de", 1: "fr", 3: "it"}
smartvote['language'] = smartvote["language"].apply(lambda x: language_mapping[x])

Preprocess corpus (arguments).

In [8]:
corpus = pd.read_json("shared_task_data/data-release-test-2/corpus.jsonl", lines=True)
corpus.head()

,argument_id,argument,stance,topic,demographic_profile
0,201900,Das Schweizer Volk hat die MEI angenommen und ...,FAVOR,Immigration,"{'gender': 'Männlich', 'age': '18-34', 'reside..."
1,201901,Eine Legalisierung von Cannabis entlasten die ...,FAVOR,Society,"{'gender': 'Männlich', 'age': '18-34', 'reside..."
2,201902,Durch die Förderung der familienergänzenden Be...,FAVOR,Welfare,"{'gender': 'Weiblich', 'age': '35-49', 'reside..."
3,201903,Ich ziehe eine Elternzeit vor. Die Zeit nach d...,AGAINST,Welfare,"{'gender': 'Weiblich', 'age': '35-49', 'reside..."
4,201904,Unser Asylrecht muss konsequent angewendet wer...,AGAINST,Immigration,"{'gender': 'Weiblich', 'age': '35-49', 'reside..."


Expand demographic profile.

In [9]:
corpus = pd.concat([corpus["demographic_profile"].apply(pd.Series),
                    corpus], axis=1)
                    

We do not need the important political issues, drop it.

In [10]:
corpus = corpus.drop(["demographic_profile", "important_political_issues"], axis=1)
corpus.head()

,gender,age,residence,civil_status,denomination,education,political_spectrum,argument_id,argument,stance,topic
0,Männlich,18-34,Land,Ledig,Christ-katholisch,Fachhochschule,Mitte und Konservativ-Liberal,201900,Das Schweizer Volk hat die MEI angenommen und ...,FAVOR,Immigration
1,Männlich,18-34,Land,Ledig,Christ-katholisch,Fachhochschule,Mitte und Konservativ-Liberal,201901,Eine Legalisierung von Cannabis entlasten die ...,FAVOR,Society
2,Weiblich,35-49,Land,Ledig,Nicht bekannt,Universität,Mitte und Konservativ,201902,Durch die Förderung der familienergänzenden Be...,FAVOR,Welfare
3,Weiblich,35-49,Land,Ledig,Nicht bekannt,Universität,Mitte und Konservativ,201903,Ich ziehe eine Elternzeit vor. Die Zeit nach d...,AGAINST,Welfare
4,Weiblich,35-49,Land,Ledig,Nicht bekannt,Universität,Mitte und Konservativ,201904,Unser Asylrecht muss konsequent angewendet wer...,AGAINST,Immigration


Filter for only 2023 arguments.

In [11]:
corpus = corpus[corpus["argument_id"].astype(str).str.startswith("2023")]

In [12]:
corpus.shape

(12758, 11)

Preprocess queries.

In [13]:
queries_train = pd.read_json("shared_task_data/data-release-test-2/baseline-queries/queries_train.jsonl", lines=True)
queries_dev = pd.read_json("shared_task_data/data-release-test-2/baseline-queries/queries_dev.jsonl", lines=True)
queries_test = pd.read_json("shared_task_data/data-release-test-2/baseline-queries/queries_test.jsonl", lines=True)

In [14]:
queries = pd.concat([queries_train, queries_dev, queries_test])

Filter for 2023 arguments.

In [15]:
queries["relevant_candidates"] = queries["relevant_candidates"].apply(lambda x: [arg for arg in x if str(arg).startswith("2023")])

Drop all queries that do not have 2023 arguments.

In [16]:
queries = queries[queries["relevant_candidates"].apply(len) != 0]

In [17]:
queries.head()

,query_id,text,relevant_candidates
0,0,Bei Ehepaaren ist die Höhe der Rente heute auf...,"[202300, 2023019, 2023087, 2023096, 20230145, ..."
1,1,Im Rahmen der BVG-Reform sollen die Renten gek...,"[202301, 2023020, 2023061, 2023097, 20230146, ..."
2,2,Befürworten Sie die Einführung einer Abgabe au...,"[2023021, 2023049, 2023062, 2023098, 20230147,..."
3,3,Sollen in Zukunft bei Pandemien die Möglichkei...,"[202302, 2023022, 2023063, 2023099, 20230138, ..."
4,4,Soll der Bund die Kompetenz zur Festlegung des...,"[202303, 2023023, 2023050, 2023064, 20230100, ..."


Create mapping for later merging.

In [18]:
query_id_mapping = {}
for i, row in queries.iterrows():
    for arg in row["relevant_candidates"]:
        if arg not in query_id_mapping:
            query_id_mapping[arg] = row["query_id"]
        else:
            print("Duplicate query_id")

We do not have more than one query_id per argument, so we can assume a 1-1 mapping

In [20]:
corpus["query_id"] = corpus["argument_id"].apply(lambda x: query_id_mapping[x] if x in query_id_mapping else None)

In [21]:
corpus.head()

,gender,age,residence,civil_status,denomination,education,political_spectrum,argument_id,argument,stance,topic,query_id
26335,Männlich,18-34,Stadt,In Partnerschaft,Evangelisch-reformiert,Universität,Mitte und Konservativ,202300,Die AHV soll unabhängig des Zivilstands ausges...,FAVOR,Welfare state & family,0
26336,Männlich,18-34,Stadt,In Partnerschaft,Evangelisch-reformiert,Universität,Mitte und Konservativ,202301,"Es ist unfair, wenn jemandem mehr Rente ausbez...",FAVOR,Welfare state & family,1
26337,Männlich,18-34,Stadt,In Partnerschaft,Evangelisch-reformiert,Universität,Mitte und Konservativ,202302,Die Massnahmen der Politik waren richtig und w...,FAVOR,Health,3
26338,Männlich,18-34,Stadt,In Partnerschaft,Evangelisch-reformiert,Universität,Mitte und Konservativ,202303,Die Kantone sind näher an der Bevölkerung und ...,AGAINST,Health,4
26339,Männlich,18-34,Stadt,In Partnerschaft,Evangelisch-reformiert,Universität,Mitte und Konservativ,202304,"Eine zu hohe Maturaquote führt dazu, dass viel...",FAVOR,Education,5


Merge to get a shared task-smartvote mapping for questions.

In [22]:
questions = pd.merge(queries, smartvote, left_on="text", right_on="question")

We need language information for mapping the correct question version to the arguments.
We thus detect the language for each argument.

In [23]:
langs = []
for i, row in corpus.iterrows():
    langs.append(langdetect.detect(row["argument"]))

In [24]:
corpus["language"] = langs

In [25]:
corpus["language"].value_counts()

language
de    10109
fr     2311
it      333
af        5
Name: count, dtype: int64

Remap the Afrikaans misclassifications.

In [26]:
corpus["language"] = corpus["language"].apply(lambda x: x if x != "af" else "de")

In [27]:
corpus["language"].value_counts()

language
de    10114
fr     2311
it      333
Name: count, dtype: int64

Merge to get the smartvote ID_question.

In [28]:
corpus = pd.merge(corpus, questions[["query_id", "ID_question"]], on="query_id")

Merge on the ids and language to create the final corpus.

In [29]:
corpus = corpus.drop(["query_id"], axis=1)
corpus = pd.merge(corpus, smartvote, on=["ID_question", "language"])

Cleanup html.

In [ ]:
tags = re.compile("<.*?>")

def clean_text(text):
    text = re.sub(tags, "", text)
    return text

corpus["info"] = corpus["info"].apply(lambda x:html.unescape(clean_text(x)) if x is not np.nan else None)
corpus["pro"] = corpus["pro"].apply(lambda x: html.unescape(clean_text(x)) if x is not np.nan else None)
corpus["contra"] = corpus["contra"].apply(lambda x: html.unescape(clean_text(x)) if x is not np.nan else None)

In [34]:
corpus["info"][0]

'Die AHV ist die Erste Säule der Altersvorsorge. Die 1. Säule soll den Rentner/-innen ein Einkommen garantieren, das die Grundbedürfnisse in der Zeit nach der Pensionierung deckt. Gegenwärtig beträgt die minimale Altersrente für eine Einzelperson monatlich 1225 Franken; die Maximalrente beläuft sich auf 2450 Franken.\nBei Ehepaaren sieht das AHV-Gesetz vor, dass die Summe der Einzelrenten den Betrag von 3675 Franken nicht überschreiten darf. Dies entspricht 150% der maximalen Einzelrente (2450 Franken). Grundsätzlich werden die Renten eines Ehepaars getrennt voneinander ausbezahlt. Überschreiten die jeweiligen Einzelrenten eines Ehepaars den Betrag von 3675 Franken, werden die Renten gekürzt. Das heißt, Ehepaare erhalten als Einzelpersonen weniger Rente, wenn sich die Summe ihrer Einzelrenten auf über 3675 Franken beläuft.'

In [36]:
corpus.columns

Index(['gender', 'age', 'residence', 'civil_status', 'denomination',
       'education', 'political_spectrum', 'argument_id', 'argument', 'stance',
       'topic', 'language', 'ID_question', 'question', 'info', 'pro',
       'contra'],
      dtype='object')

In [35]:
corpus.to_csv("corpus.csv", index=False)